In [ ]:
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar
from IPython.display import display
import ipywidgets as widgets
import traitlets

import gc
import time

controller = widgets.Controller(index=0)
display(controller)



In [2]:
car = NvidiaRacecar()

car.throttle_gain = 0.5
car.steering_offset = 0
car.steering_gain = -0.65

car.steering = 0
car.throttle = 0

In [4]:
steering_link = traitlets.dlink(
    (controller.axes[0], 'value'), 
    (car, 'steering'), 
    transform=lambda x: -x
)

throttle_link = traitlets.dlink(
    (controller.buttons[7], 'value'), 
    (car, 'throttle'), 
    transform=lambda x: -x
)

brake_link = traitlets.dlink(
    (controller.buttons[6], 'value'), 
    (car, 'throttle'), 
    transform=lambda x: x
)

In [5]:

# --- Automatyczne sprzątanie ---
if 'camera' in globals():
    print("Zamykanie poprzedniej sesji kamery...")
    try:
        camera.running = False
        camera.unobserve_all()
        time.sleep(1.0) # Daj chwilę sterownikowi nvargus
        del camera
        gc.collect()
    except:
        pass

# --- Start nowej sesji ---
try:
    # Kamera sprzętowo
    camera = CSICamera(width=224, height=224, capture_fps=30)
    
    image_widget = widgets.Image(format='jpeg', width=224, height=224)

    def update_image(change):
        image = change['new']
        image_widget.value = bgr8_to_jpeg(image)

    camera.observe(update_image, names='value')
    camera.running = True

    display(image_widget)
    print("Kamera działa! Jeśli chcesz przestać, użyj camera.running = False")
except Exception as e:
    print(f"Błąd: {e}")
    print("Jeśli nic nie pomaga, odpal w terminalu: sudo systemctl restart nvargus-daemon")

Image(value=b'', format='jpeg', height='224', width='224')

Kamera działa! Jeśli chcesz przestać, użyj camera.running = False
